# IntelliQuery-RAG — Step-by-Step Tutorial 🧠⚡

This notebook walks you through the **complete pipeline** of the IntelliQuery-RAG system,
from loading a PDF to asking complex multi-step questions.

We'll use Frank Herbert's *Dune* as our document corpus.

## Prerequisites
- A copy of the Dune PDF placed in the project root
- An OpenAI API key in your `.env` file
- `pip install -e ".[dev]"`

## 1. Configuration & Setup

In [1]:
import sys
sys.path.insert(0, '..')

from intelliquery.config import load_settings

settings = load_settings()
print(f"LLM Provider: {settings.llm.provider}")
print(f"Model: {settings.llm.model_name}")
print(f"Vector store dir: {settings.vector_store.persist_directory}")
print(f"API key loaded: {'✅' if settings.openai_api_key else '❌'}")

LLM Provider: openai
Model: gpt-4o
Vector store dir: ./data
API key loaded: ✅


## 2. Load and Split the PDF

We extract the full text from the Dune PDF, then split it into sections
based on the "BOOK ONE / TWO / THREE" structure.

In [2]:
from intelliquery.processing.pdf_loader import load_and_split_pdf

# 📝 Update this path to your Dune PDF
PDF_PATH = "../dune.pdf"

sections = load_and_split_pdf(PDF_PATH)
print(f"Found {len(sections)} sections:\n")
for sec in sections:
    preview = sec.page_content[:100].replace('\n', ' ')
    print(f"  Section {sec.metadata['section']}: {sec.metadata['header']}")
    print(f"    Preview: {preview}...")
    print(f"    Length: {len(sec.page_content):,} chars\n")

ModuleNotFoundError: No module named 'langchain.docstore'

## 3. Generate Section Summaries

Each section is summarised by the LLM to create high-level overviews
that help with broad retrieval queries.

In [ ]:
from intelliquery.providers.openai_provider import OpenAILLMProvider
from intelliquery.processing.summarizer import summarize_all_sections

provider = OpenAILLMProvider()
llm = provider.build_chat_model(
    model_name=settings.llm.model_name,
    temperature=settings.llm.temperature,
)

summaries = summarize_all_sections(sections, llm, verbose=True)

print(f"\n✅ Generated {len(summaries)} summaries")
for s in summaries:
    print(f"\n--- Section {s.metadata['section']} Summary ---")
    print(s.page_content[:300] + "...")

## 4. Build Vector Stores

We create three FAISS indexes:
1. **Chunks** — raw text split into overlapping chunks
2. **Summaries** — LLM-generated section summaries
3. **Quotes** — notable passages and dialogues (using the raw sections for now)

In [ ]:
from intelliquery.retrievers.vector_store import FAISSStoreManager

embeddings = provider.build_embeddings(model_name=settings.embeddings.model_name)
store_mgr = FAISSStoreManager(embeddings=embeddings, persist_dir=settings.vector_store.persist_directory)

# Build chunks index (splits sections into overlapping chunks)
print("📦 Building chunks index...")
store_mgr.build_from_documents(
    sections,
    index_name=settings.vector_store.chunks_index,
    chunk_size=settings.processing.chunk_size,
    chunk_overlap=settings.processing.chunk_overlap,
    split=True,
)

# Build summaries index (no splitting needed — one doc per summary)
print("📦 Building summaries index...")
store_mgr.build_from_documents(
    summaries,
    index_name=settings.vector_store.summaries_index,
    split=False,
)

# Build quotes index (using raw sections as a proxy for quotes)
print("📦 Building quotes index...")
store_mgr.build_from_documents(
    sections,
    index_name=settings.vector_store.quotes_index,
    chunk_size=500,
    chunk_overlap=50,
    split=True,
)

print("\n✅ All vector stores built and persisted!")

## 5. Test Retrieval

Let's verify the retrieval pipeline works before running the full agent.

In [ ]:
from intelliquery.retrievers.multi_retriever import DocumentRetriever

retriever = DocumentRetriever(
    store_manager=store_mgr,
    chunks_index=settings.vector_store.chunks_index,
    summaries_index=settings.vector_store.summaries_index,
    quotes_index=settings.vector_store.quotes_index,
    chunk_k=settings.retriever.chunk_top_k,
    summary_k=settings.retriever.summary_top_k,
    quotes_k=settings.retriever.quotes_top_k,
)

test_query = "What is the spice melange and why is it valuable?"
result = retriever.retrieve(test_query, verbose=True)

print(f"\n📄 Merged context ({len(result.merged)} chars):")
print(result.merged[:500] + "...")

## 6. Run the Full Agent

Now let's build the state-graph and ask a complex multi-step question!

In [ ]:
from intelliquery.agents.graph import build_agent_graph

agent = build_agent_graph(settings=settings, retriever=retriever)

complex_question = (
    "Why did the Duke accept the posting to Arrakis despite knowing "
    "it was a trap, and how did his son eventually turn the tables "
    "on the conspirators?"
)

print(f"❓ Question: {complex_question}\n")
print("=" * 60)

result = agent.invoke({"query": complex_question})

print("\n" + "=" * 60)
print("\n💡 FINAL ANSWER:")
print(result["final_response"])

## 7. Inspect the Reasoning Trace

Let's look at exactly how the agent arrived at its answer.

In [ ]:
print("🧠 REASONING TRACE:\n")
for i, step in enumerate(result.get("reasoning_trace", []), 1):
    print(f"--- Step {i} ---")
    print(step)
    print()

## 8. Evaluate the Response (Optional)

If you have a ground-truth answer, you can evaluate the agent's response
using Ragas metrics.

In [ ]:
from intelliquery.evaluation.metrics import evaluate_response

ground_truth = (
    "Duke Leto accepted the Arrakis posting because controlling the spice "
    "would give House Atreides political leverage, and he hoped to ally "
    "with the Fremen. His son Paul eventually united the Fremen, gained "
    "control of spice production, and challenged the Emperor directly."
)

eval_result = evaluate_response(
    question=complex_question,
    generated_answer=result["final_response"],
    ground_truth=ground_truth,
    contexts=[result.get("accumulated_evidence", "")],
)

print("📊 EVALUATION SCORES:\n")
print(eval_result.summary())

## 🎉 Done!

You've successfully:
1. Loaded and split a PDF into sections
2. Generated LLM summaries of each section
3. Built three FAISS vector indexes
4. Ran the full graph-orchestrated agent on a complex question
5. Inspected the reasoning trace
6. Evaluated the response with Ragas metrics

### Next Steps
- Launch the Gradio UI: `python -m ui.app`
- Try different questions and observe how the agent plans differently
- Swap the LLM provider in `config.yaml`
- Add your own documents!